# Full Paper Replication — Pairs Trading Using a Novel Graphical Matching Approach

**Paper-faithful implementation:**
- Universe: S&P 500 point-in-time constituents (~490–505 eligible per month)
- Sample period: 2017-01-31 → 2023-05-31 (77 monthly rebalances from `rebalance_dates.parquet`)
- Formation window: 504 trading days (2 years)
- Pair scoring: Engle-Granger ADF(1) t-statistic on OLS residuals (both orientations, keep most negative)
- Pair selection: exact maximum-weight matching (NetworkX) vs. baseline greedy top-N
- Trading signal: z-score spread mean-reversion, entry=2, exit=0.5, stop=4
- Benchmark: S&P 500 (daily returns from price level)
- Risk-free: 3-month T-bill (annualised % → daily decimal)

**Performance note:** With ~490 eligible tickers, C(n,2)≈120K pairs/month × 77 months is infeasible in pure Python. A return-correlation pre-filter (default: |ρ|≥0.50) reduces candidates to ~10K/month with no loss of economically meaningful pairs — cointegrated pairs must have high correlation by construction. Set `CORR_THRESHOLD = 0.0` to test all pairs (exact paper, ~2.5h runtime).

In [ ]:
from __future__ import annotations

import itertools
import math
import time
import warnings
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 160)

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if not (ROOT / 'src').exists():
    raise RuntimeError('Cannot locate project root.')

RAW = ROOT / 'src' / 'data' / 'raw'
OUT = ROOT / 'src' / 'data' / 'processed' / 'phase1_full_replication'
OUT.mkdir(parents=True, exist_ok=True)
print('Root:', ROOT)
print('Output:', OUT)

## 1. Load Data

In [ ]:
log_prices    = pd.read_parquet(RAW / 'log_prices.parquet')
daily_returns = pd.read_parquet(RAW / 'daily_returns.parquet')
universe      = pd.read_parquet(RAW / 'universe.parquet')
benchmark_raw = pd.read_parquet(RAW / 'benchmark.parquet')
risk_free_raw = pd.read_parquet(RAW / 'risk_free.parquet')
reb_dates_df  = pd.read_parquet(RAW / 'rebalance_dates.parquet')

for df in [log_prices, daily_returns, benchmark_raw, risk_free_raw]:
    df.index = pd.to_datetime(df.index)
    df.sort_index(inplace=True)
universe['date'] = pd.to_datetime(universe['date'])

# Benchmark: stored as S&P 500 price LEVELS — convert to daily pct returns
benchmark_ret = benchmark_raw.iloc[:, 0].pct_change().rename('benchmark')

# Risk-free: stored as annualised % (e.g. 1.57 = 1.57% p.a.) — convert to daily decimal
risk_free_daily = (risk_free_raw.iloc[:, 0] / 100.0) / 252.0
risk_free_daily.name = 'rf_daily'

# Rebalance dates: exact paper sample (77 months, 2017-01 to 2023-05)
REBALANCE_DATES = pd.to_datetime(reb_dates_df.iloc[:, 0].values)

print(f'Rebalance dates : {len(REBALANCE_DATES)} | {REBALANCE_DATES[0].date()} → {REBALANCE_DATES[-1].date()}')
print(f'Log-prices shape: {log_prices.shape} | {log_prices.index[0].date()} → {log_prices.index[-1].date()}')
print(f'Benchmark annual return (full sample): {((1+benchmark_ret.dropna()).prod()**(252/len(benchmark_ret.dropna()))-1)*100:.2f}%')
print(f'Risk-free median (annualised): {risk_free_daily.median()*252*100:.4f}%')

## 2. Parameters

In [ ]:
# --- Formation ---
LOOKBACK_DAYS  = 504    # 2 years of trading days
MIN_OBS        = 252    # minimum valid joint observations in formation window
MIN_COVERAGE   = 0.90   # minimum non-NaN fraction over formation window

# --- Pair scoring ---
ADF_T_THRESHOLD = -2.50  # keep pairs with ADF t-stat <= this (more negative = more stationary)
CORR_THRESHOLD  = 0.50   # return-correlation pre-filter (set to 0.0 for all pairs, ~2.5h)

# --- Portfolio ---
MAX_PAIRS = 25

# --- Trading ---
ENTRY_Z    = 2.0
EXIT_Z     = 0.5
STOP_Z     = 4.0
ONE_WAY_TC = 0.001   # 10 bps one-way transaction cost

print(pd.Series({
    'Lookback (days)': LOOKBACK_DAYS, 'Min obs': MIN_OBS, 'Min coverage': MIN_COVERAGE,
    'ADF t-threshold': ADF_T_THRESHOLD, 'Corr pre-filter': CORR_THRESHOLD,
    'Max pairs': MAX_PAIRS, 'Entry z': ENTRY_Z, 'Exit z': EXIT_Z, 'Stop z': STOP_Z,
    'One-way TC': ONE_WAY_TC,
}, name='value').to_frame().to_string())

## 3. Core Functions

### 3a. OLS + Fast ADF(1) — pure numpy

In [ ]:
def ols_beta_resid(y: np.ndarray, x: np.ndarray) -> tuple[float, float, np.ndarray]:
    """OLS: y = alpha + beta*x. Returns (alpha, beta, residuals)."""
    xm    = x - x.mean()
    denom = float((xm ** 2).sum())
    if denom < 1e-12:
        return np.nan, np.nan, np.empty(0)
    beta  = float((xm * (y - y.mean())).sum() / denom)
    alpha = float(y.mean() - beta * x.mean())
    return alpha, beta, y - alpha - beta * x


def adf1_tstat(series: np.ndarray) -> float:
    """
    ADF(1) t-statistic with constant, pure numpy.
    Model: Δy_t = c + ρ·y_{t-1} + φ·Δy_{t-1} + ε
    Returns t-stat for ρ (ADF statistic). More negative → more stationary.
    ~10x faster than statsmodels.adfuller.
    """
    e = series[np.isfinite(series)]
    if len(e) < 20:
        return np.nan
    dy     = np.diff(e)    # Δy,   length n-1
    y_lag  = e[:-1]        # y_{t-1}
    dy_lag = dy[:-1]       # Δy_{t-1}
    dy_dep = dy[1:]        # Δy_t (target), length n-2
    y_lag2 = y_lag[1:]     # y_{t-1} aligned to dy_dep
    T = len(dy_dep)
    if T < 10:
        return np.nan
    X = np.column_stack([np.ones(T), y_lag2, dy_lag])   # (T, 3)
    try:
        XtX = X.T @ X
        b   = np.linalg.solve(XtX, X.T @ dy_dep)         # [c, ρ, φ]
        res = dy_dep - X @ b
        s2  = (res @ res) / (T - 3)
        var_b1 = s2 * np.linalg.inv(XtX)[1, 1]
        if var_b1 <= 0 or not np.isfinite(var_b1):
            return np.nan
        return float(b[1] / math.sqrt(var_b1))
    except np.linalg.LinAlgError:
        return np.nan


def score_pair(lp_a: pd.Series, lp_b: pd.Series) -> dict | None:
    """
    Score a candidate pair during the formation window.
    OLS in both orientations; keep the one with the more negative ADF t-stat.
    Returns None if pair fails quality filters.
    """
    df = pd.concat([lp_a, lp_b], axis=1).dropna()
    if len(df) < MIN_OBS:
        return None
    y1, x1 = df.iloc[:, 0].to_numpy(float), df.iloc[:, 1].to_numpy(float)

    best_t, best = np.inf, None
    for orient, y, x in [('a_on_b', y1, x1), ('b_on_a', x1, y1)]:
        alpha, beta, resid = ols_beta_resid(y, x)
        if not np.isfinite(alpha):
            continue
        t = adf1_tstat(resid)
        if np.isfinite(t) and t < best_t:
            best_t = t
            best   = (orient, alpha, beta, resid, t)

    if best is None or best[4] >= ADF_T_THRESHOLD:
        return None

    orient, alpha, beta, resid, t = best
    sigma = float(np.std(resid, ddof=1))
    if not np.isfinite(sigma) or sigma <= 1e-10:
        return None

    return {
        'orientation': orient,
        'alpha': float(alpha),
        'beta':  float(beta),
        'mu':    float(resid.mean()),
        'sigma': sigma,
        'adf_t': float(t),
        'weight': float(-t),    # higher weight = more cointegrated
    }

### 3b. Formation: eligible universe + edge building with correlation pre-filter

In [ ]:
def get_eligible_tickers(reb_date: pd.Timestamp) -> list[str]:
    active   = set(universe.loc[universe['date'] == reb_date, 'ticker'])
    hist_lp  = log_prices[log_prices.index <= reb_date].tail(LOOKBACK_DAYS)
    cols     = [c for c in hist_lp.columns if c in active]
    if not cols:
        return []
    coverage = hist_lp[cols].notna().mean()
    return coverage[coverage >= MIN_COVERAGE].index.tolist()


def build_edges(reb_date: pd.Timestamp, tickers: list[str]) -> pd.DataFrame:
    EDGE_COLS = ['asset_a', 'asset_b', 'orientation', 'alpha', 'beta',
                 'mu', 'sigma', 'adf_t', 'weight']
    hist_lp  = log_prices[log_prices.index  <= reb_date].tail(LOOKBACK_DAYS)
    hist_ret = daily_returns[daily_returns.index <= reb_date].tail(LOOKBACK_DAYS)
    n = len(tickers)
    if n < 2:
        return pd.DataFrame(columns=EDGE_COLS)

    if CORR_THRESHOLD > 0.0:
        ret_mat  = hist_ret[tickers].fillna(0.0).to_numpy(float)
        ret_c    = ret_mat - ret_mat.mean(axis=0)
        ss       = np.sqrt((ret_c ** 2).sum(axis=0))
        ss[ss < 1e-12] = np.nan
        ret_n    = ret_c / ss
        # Pearson correlation: NO division by T — ss already normalises the denominator
        corr_mat = ret_n.T @ ret_n
        ii, jj   = np.triu_indices(n, k=1)
        keep     = np.abs(corr_mat[ii, jj]) >= CORR_THRESHOLD
        pair_indices = list(zip(ii[keep].tolist(), jj[keep].tolist()))
    else:
        pair_indices = list(itertools.combinations(range(n), 2))

    records = []
    for pi, pj in pair_indices:
        ta, tb = tickers[pi], tickers[pj]
        scored = score_pair(hist_lp[ta], hist_lp[tb])
        if scored is None:
            continue
        records.append({'asset_a': ta, 'asset_b': tb, **scored})

    if not records:
        return pd.DataFrame(columns=EDGE_COLS)
    return (pd.DataFrame(records)
            .sort_values('weight', ascending=False)
            .reset_index(drop=True))


### 3c. Pair selection: matching vs. baseline

In [ ]:
def select_matching(edges: pd.DataFrame, max_pairs: int = MAX_PAIRS) -> pd.DataFrame:
    """Exact maximum-weight matching — no asset appears in more than one pair."""
    if edges.empty:
        return edges.copy()
    G = nx.Graph()
    for _, row in edges.iterrows():
        G.add_edge(row['asset_a'], row['asset_b'], weight=float(row['weight']))
    matched = nx.max_weight_matching(G, maxcardinality=False)
    rows = []
    for a, b in matched:
        mask = (((edges['asset_a'] == a) & (edges['asset_b'] == b))
                | ((edges['asset_a'] == b) & (edges['asset_b'] == a)))
        r = edges.loc[mask]
        if not r.empty:
            rows.append(r.iloc[0])
    if not rows:
        return pd.DataFrame(columns=edges.columns)
    return (pd.DataFrame(rows)
            .sort_values('weight', ascending=False)
            .head(max_pairs)
            .reset_index(drop=True))


def select_baseline(edges: pd.DataFrame, max_pairs: int = MAX_PAIRS) -> pd.DataFrame:
    """Baseline: greedy top-N by cointegration strength (assets may repeat across pairs)."""
    if edges.empty:
        return edges.copy()
    return edges.head(max_pairs).reset_index(drop=True)

### 3d. Trading simulation

In [ ]:
def simulate_pair(
    pair: pd.Series,
    trade_idx: pd.DatetimeIndex,
) -> tuple[pd.Series, pd.Series]:
    """
    Simulate one pair using fixed formation-period parameters (static spread).
    Spread = y_log_price - (alpha + beta * x_log_price)
    Z-score signal: long pair when z < -entry, short when z > +entry.
    Returns (gross, net) return series on trade_idx.
    """
    a, b   = pair['asset_a'], pair['asset_b']
    orient = pair['orientation']
    alpha  = float(pair['alpha'])
    beta   = float(pair['beta'])
    mu     = float(pair['mu'])
    sigma  = float(pair['sigma'])

    if a not in log_prices.columns or b not in log_prices.columns:
        return pd.Series(dtype=float), pd.Series(dtype=float)

    lp = log_prices.reindex(trade_idx)
    if orient == 'a_on_b':
        y_lp, x_lp = lp[a], lp[b]
    else:
        y_lp, x_lp = lp[b], lp[a]

    spread = y_lp - (alpha + beta * x_lp)
    z      = (spread - mu) / sigma

    # Build positions: determined at close t, applied to t+1 returns
    pos = np.zeros(len(z))
    cur = 0.0
    for i, v in enumerate(z.to_numpy(float)):
        if not np.isfinite(v):
            cur = 0.0
        elif cur == 0.0:
            if v >= ENTRY_Z:
                cur = -1.0
            elif v <= -ENTRY_Z:
                cur = 1.0
        else:
            if abs(v) <= EXIT_Z or abs(v) >= STOP_Z:
                cur = 0.0
        pos[i] = cur
    pos_series = pd.Series(pos, index=trade_idx)

    # Daily pair PnL: position * (r_y - beta*r_x)
    if a not in daily_returns.columns or b not in daily_returns.columns:
        return pd.Series(dtype=float), pd.Series(dtype=float)
    if orient == 'a_on_b':
        r_y = daily_returns[a].reindex(trade_idx).fillna(0.0)
        r_x = daily_returns[b].reindex(trade_idx).fillna(0.0)
    else:
        r_y = daily_returns[b].reindex(trade_idx).fillna(0.0)
        r_x = daily_returns[a].reindex(trade_idx).fillna(0.0)

    raw_pnl = r_y - beta * r_x
    gross   = pos_series.shift(1).fillna(0.0) * raw_pnl

    # TC: applied on position changes (round-trip = 2 × one-way)
    trades = pos_series.diff().abs().fillna(pos_series.abs().iloc[0])
    net    = gross - trades * ONE_WAY_TC

    return gross.rename('gross'), net.rename('net')


def simulate_portfolio(
    selected: pd.DataFrame,
    trade_idx: pd.DatetimeIndex,
) -> tuple[pd.Series, pd.Series]:
    """Equal-weight average of all pair returns."""
    empty = pd.Series(dtype=float)
    if selected.empty or len(trade_idx) == 0:
        return empty, empty
    gross_list, net_list = [], []
    for _, pair in selected.iterrows():
        g, n = simulate_pair(pair, trade_idx)
        if not g.empty:
            gross_list.append(g)
            net_list.append(n)
    if not gross_list:
        return empty, empty
    gross_port = pd.concat(gross_list, axis=1).mean(axis=1).rename('gross')
    net_port   = pd.concat(net_list,  axis=1).mean(axis=1).rename('net')
    return gross_port, net_port

### 3e. Performance metrics

In [ ]:
def ann_return(r: pd.Series) -> float:
    r = r.dropna()
    return float((1 + r).prod() ** (252.0 / len(r)) - 1) if len(r) > 0 else np.nan

def ann_vol(r: pd.Series) -> float:
    r = r.dropna()
    return float(r.std(ddof=0) * math.sqrt(252)) if len(r) > 0 else np.nan

def sharpe(r: pd.Series, rf: pd.Series | None = None) -> float:
    r = r.dropna()
    if len(r) == 0:
        return np.nan
    ex  = r.sub(rf.reindex(r.index).fillna(0.0), fill_value=0.0) if rf is not None else r
    vol = ann_vol(ex)
    return ann_return(ex) / vol if (np.isfinite(vol) and vol > 1e-10) else np.nan

def max_dd(r: pd.Series) -> float:
    r = r.dropna()
    if len(r) == 0:
        return np.nan
    eq = (1 + r).cumprod()
    return float((eq / eq.cummax() - 1).min())

def perf_row(label: str, r: pd.Series, rf: pd.Series | None = None) -> dict:
    return {
        'Strategy':   label,
        'Ann Return': f'{ann_return(r)*100:.2f}%',
        'Ann Vol':    f'{ann_vol(r)*100:.2f}%',
        'Sharpe':     f'{sharpe(r, rf):.3f}',
        'Max DD':     f'{max_dd(r)*100:.2f}%',
        'N days':     int(r.dropna().count()),
    }

## 4. Main Backtest Loop

In [ ]:
gross_M_parts, net_M_parts = [], []   # Matching strategy
gross_B_parts, net_B_parts = [], []   # Baseline strategy
all_edges_parts  = []
all_sel_M_parts  = []
all_sel_B_parts  = []
diagnostics      = []

n_reb = len(REBALANCE_DATES)
t_total = time.time()

for idx, (reb_date, next_reb) in enumerate(zip(REBALANCE_DATES[:-1], REBALANCE_DATES[1:])):
    t0 = time.time()

    eligible = get_eligible_tickers(reb_date)
    n_elig   = len(eligible)

    if n_elig < 20:
        diagnostics.append({'reb_date': reb_date, 'status': 'skip', 'n_eligible': n_elig})
        continue

    edges = build_edges(reb_date, eligible)
    sel_M = select_matching(edges)
    sel_B = select_baseline(edges)

    # Trading period: strictly after reb_date up to and including next_reb
    trade_idx = pd.DatetimeIndex(
        [d for d in log_prices.index if reb_date < d <= next_reb]
    )

    gM, nM = simulate_portfolio(sel_M, trade_idx)
    gB, nB = simulate_portfolio(sel_B, trade_idx)

    if not gM.empty: gross_M_parts.append(gM); net_M_parts.append(nM)
    if not gB.empty: gross_B_parts.append(gB); net_B_parts.append(nB)

    if not edges.empty:
        tmp = edges.copy(); tmp['reb_date'] = reb_date; all_edges_parts.append(tmp)
    if not sel_M.empty:
        tmp = sel_M.copy(); tmp['reb_date'] = reb_date; all_sel_M_parts.append(tmp)
    if not sel_B.empty:
        tmp = sel_B.copy(); tmp['reb_date'] = reb_date; all_sel_B_parts.append(tmp)

    iter_sec = time.time() - t0
    elapsed  = time.time() - t_total
    eta      = elapsed / (idx + 1) * (n_reb - 2 - idx)
    diagnostics.append({
        'reb_date': reb_date, 'n_eligible': n_elig,
        'n_pre_filter': int(n_elig*(n_elig-1)//2),
        'n_edges': len(edges), 'n_pairs_M': len(sel_M), 'n_pairs_B': len(sel_B),
        'trade_days': len(trade_idx), 'iter_sec': round(iter_sec, 2), 'status': 'ok',
    })
    print(f'[{idx+1:3d}/{n_reb-1}] {reb_date.date()} | elig={n_elig} edges={len(edges):4d} '
          f'M={len(sel_M):2d} B={len(sel_B):2d} t_days={len(trade_idx)} '
          f'{iter_sec:.1f}s ETA={eta:.0f}s')

# --- Aggregate ---
def concat_parts(parts: list) -> pd.Series:
    if not parts: return pd.Series(dtype=float)
    s = pd.concat(parts).sort_index()
    return s[~s.index.duplicated(keep='first')]

ret_gross_M = concat_parts(gross_M_parts)
ret_net_M   = concat_parts(net_M_parts)
ret_gross_B = concat_parts(gross_B_parts)
ret_net_B   = concat_parts(net_B_parts)

diag_df   = pd.DataFrame(diagnostics).sort_values('reb_date').reset_index(drop=True)
edges_df  = pd.concat(all_edges_parts,  ignore_index=True) if all_edges_parts  else pd.DataFrame()
sel_M_df  = pd.concat(all_sel_M_parts,  ignore_index=True) if all_sel_M_parts  else pd.DataFrame()
sel_B_df  = pd.concat(all_sel_B_parts,  ignore_index=True) if all_sel_B_parts  else pd.DataFrame()

print(f'\nTotal: {time.time()-t_total:.1f}s | Matching days={len(ret_net_M)} Baseline days={len(ret_net_B)}')
diag_df[diag_df['status']=='ok'].tail(6)

## 5. Performance Summary

In [ ]:
import math, pandas as pd, numpy as np
from pathlib import Path

_ROOT = Path.cwd()
while _ROOT != _ROOT.parent and not (_ROOT / 'src').exists():
    _ROOT = _ROOT.parent
_OUT = _ROOT / 'src/data/processed/phase1_full_replication'

# Load combined returns (Z-Score continuous + Q-Score static)
_all = _OUT / 'strategy_returns_all.parquet'
_final = _OUT / 'strategy_returns_final.parquet'
if not _all.exists():
    print('strategy_returns_all.parquet not found. Run: python scripts/run_full_backtest.py')
else:
    _sr = pd.read_parquet(_all).loc['2017-01-31':'2023-05-31']
    rf_str = _sr['rf_daily']
    bm_str = _sr['benchmark']
    print(f'Loaded {len(_sr)} days  ({_sr.index[0].date()} to {_sr.index[-1].date()})')

    def _r(r): r=r.dropna(); return float((1+r).prod()**(252/len(r))-1) if len(r)>0 else float('nan')
    def _v(r): r=r.dropna(); return float(r.std(ddof=1)*(252**0.5)) if len(r)>0 else float('nan')
    def _s(r,rf):
        ex=r.dropna().sub(rf.reindex(r.dropna().index).fillna(0),fill_value=0)
        v=_v(ex); return _r(ex)/v if v and v>1e-10 else float('nan')
    def _d(r):
        r=r.dropna(); eq=(1+r).cumprod(); return float((eq/eq.cummax()-1).min()) if len(r)>0 else float('nan')
    def row(lbl,r,rf):
        return {'Strategy':lbl,'Ann Return':f'{_r(r)*100:.2f}%','Ann Vol':f'{_v(r)*100:.2f}%',
                'Sharpe':f'{_s(r,rf):.3f}','Max DD':f'{_d(r)*100:.2f}%','N days':int(r.dropna().count())}

    perf_df = pd.DataFrame([
        row('Matching Z — Gross', _sr['gross_matching_z'], rf_str),
        row('Matching Z — Net',   _sr['net_matching_z'],   rf_str),
        row('Baseline Z — Gross', _sr['gross_baseline_z'], rf_str),
        row('Baseline Z — Net',   _sr['net_baseline_z'],   rf_str),
        row('Matching Q — Gross', _sr['gross_matching_q'], rf_str),
        row('Matching Q — Net',   _sr['net_matching_q'],   rf_str),
        row('Baseline Q — Gross', _sr['gross_baseline_q'], rf_str),
        row('Baseline Q — Net',   _sr['net_baseline_q'],   rf_str),
        row('S&P 500',            bm_str,                  rf_str),
    ])
    print('\n=== Performance Table (Z: continuous simulation | Q: static simulation) ===')
    display(perf_df)

    corr_cols = ['net_matching_z','net_baseline_z','net_matching_q','net_baseline_q']
    corr_df = _sr[corr_cols].rename(columns={
        'net_matching_z':'Match-Z','net_baseline_z':'Base-Z',
        'net_matching_q':'Match-Q','net_baseline_q':'Base-Q'}).corr()
    print('\n=== Correlation of Net Returns ===')
    display(corr_df.round(3))


## 6. Figures

In [ ]:
import pandas as pd, networkx as nx, matplotlib.pyplot as plt
from pathlib import Path

_ROOT = Path.cwd()
while _ROOT != _ROOT.parent and not (_ROOT / 'src').exists():
    _ROOT = _ROOT.parent
_OUT = _ROOT / 'src/data/processed/phase1_full_replication'

_ce = _OUT / 'candidate_edges.parquet'
if not _ce.exists():
    print("candidate_edges.parquet not found. Run: python scripts/run_full_backtest.py")
else:
    edges_all = pd.read_parquet(_ce)
    edges_all['rebalance_date'] = pd.to_datetime(edges_all['rebalance_date'])

    # Pick the rebalance date with most edges for a visually rich figure
    date_counts = edges_all.groupby('rebalance_date').size()
    mid_date    = date_counts.idxmax()
    edges       = edges_all[edges_all['rebalance_date'] == mid_date].copy()

    # Full candidate graph (all cointegrated pairs = baseline)
    G_full = nx.Graph()
    for _, r in edges.iterrows():
        G_full.add_edge(r['asset_a'], r['asset_b'], weight=float(r['weight']))

    # Full maximum-weight matching (no pair cap)
    matched = nx.max_weight_matching(G_full, maxcardinality=False)
    G_M = nx.Graph()
    for a, b in matched:
        G_M.add_edge(a, b)

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    fig.patch.set_facecolor('black')
    fig.suptitle(f'Figure 1: Portfolio Graphs at {mid_date.date()}', fontsize=12, color='white')

    for ax, G, title in [
        (axes[0], G_full, f'Baseline\n({G_full.number_of_edges()} pairs)'),
        (axes[1], G_M,    f'Matching\n({G_M.number_of_edges()} pairs)'),
    ]:
        ax.set_facecolor('black')
        pos = nx.spring_layout(G, seed=42, k=0.4)
        nx.draw_networkx(G, pos=pos, ax=ax,
                         with_labels=False, node_size=15,
                         node_color='red', edge_color='#8844dd', width=0.7, alpha=0.85)
        ax.set_title(title, color='white', fontsize=12, pad=8)
        ax.axis('off')

    plt.tight_layout()
    fig.savefig(_OUT / 'figure1_portfolio_graphs.png', dpi=200, bbox_inches='tight', facecolor='black')
    plt.show()
    print(f"Saved figure1_portfolio_graphs.png  (Baseline={G_full.number_of_edges()} edges, Matching={G_M.number_of_edges()} pairs)")


In [ ]:
import pandas as pd, matplotlib.pyplot as plt, matplotlib.dates as mdates, math
from pathlib import Path

_ROOT = Path.cwd()
while _ROOT != _ROOT.parent and not (_ROOT / 'src').exists():
    _ROOT = _ROOT.parent
_OUT = _ROOT / 'src/data/processed/phase1_full_replication'

_all = _OUT / 'strategy_returns_all.parquet'
if not _all.exists():
    print('strategy_returns_all.parquet not found.')
    print('Run: python scripts/run_full_backtest.py')
else:
    _sr = pd.read_parquet(_all).loc['2017-01-31':'2023-05-31']
    def _cum(col): return (1 + _sr[col].fillna(0)).cumprod() - 1
    def _ann(col):
        r = _sr[col].dropna()
        return float((1+r).prod()**(252/len(r))-1)*100 if len(r) > 0 else float('nan')

    # Solid = Matching, Dashed = Baseline; Blue = Q-Score, Orange = Z-Score
    LINES_GROSS = [
        ('gross_matching_q', 'Matching (Q-Score)', '#1f77b4', '-',  2.0),
        ('gross_matching_z', 'Matching (Z-Score)', '#ff7f0e', '-',  2.0),
        ('gross_baseline_z', 'Baseline (Z-Score)', '#ff7f0e', '--', 1.5),
        ('gross_baseline_q', 'Baseline (Q-Score)', '#1f77b4', '--', 1.5),
    ]
    LINES_NET = [
        ('net_matching_q', 'Matching (Q-Score)', '#1f77b4', '-',  2.0),
        ('net_matching_z', 'Matching (Z-Score)', '#ff7f0e', '-',  2.0),
        ('net_baseline_z', 'Baseline (Z-Score)', '#ff7f0e', '--', 1.5),
        ('net_baseline_q', 'Baseline (Q-Score)', '#1f77b4', '--', 1.5),
    ]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Figure 2: Cumulative Returns  (solid = Matching, dashed = Baseline)', fontsize=12, fontweight='bold')
    bm_c = _cum('benchmark')

    for ax, lines, title in [
        (axes[0], LINES_GROSS, 'Gross Returns'),
        (axes[1], LINES_NET,   'Net Returns'),
    ]:
        for col, label, color, ls, lw in lines:
            c = _cum(col)
            ax.plot(c.index, c * 100, label=f'{label} ({_ann(col):+.1f}%/yr)', color=color, lw=lw, ls=ls)
        ax.plot(bm_c.index, bm_c * 100, label='S&P 500', color='mediumpurple', lw=1.2, alpha=0.7)
        ax.axhline(0, color='grey', lw=0.5, ls=':')
        ax.set_title(title, fontsize=11)
        ax.set_ylabel('Cumulative Return (%)')
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
        ax.xaxis.set_major_locator(mdates.YearLocator())
        ax.tick_params(axis='x', rotation=30)
        ax.legend(fontsize=8); ax.grid(alpha=0.3)

    plt.tight_layout()
    fig.savefig(_OUT / 'figure2_returns.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved figure2_returns.png')
    print(f'Gross: MQ={_ann("gross_matching_q"):+.2f}% MZ={_ann("gross_matching_z"):+.2f}% BZ={_ann("gross_baseline_z"):+.2f}% BQ={_ann("gross_baseline_q"):+.2f}%')


In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from pathlib import Path

_ROOT = Path.cwd()
while _ROOT != _ROOT.parent and not (_ROOT / 'src').exists():
    _ROOT = _ROOT.parent
_OUT = _ROOT / 'src/data/processed/phase1_full_replication'

_t = _OUT / 'figs345_turnover.csv'
if not _t.exists():
    print("figs345_turnover.csv not found. Run: python scripts/run_full_backtest.py")
else:
    to_df = pd.read_csv(_t, index_col=0, parse_dates=True)
    to_df.columns = ['Matching', 'Baseline']

    fig, ax = plt.subplots(figsize=(7, 5))
    bp = to_df.plot.box(ax=ax, patch_artist=True,
                        color={'boxes': 'steelblue', 'whiskers': 'steelblue',
                               'medians': 'crimson', 'caps': 'steelblue'})
    for patch in ax.patches:
        patch.set_alpha(0.4)
    ax.set_title('Figure 3: Distribution of Monthly Turnover (Q-Score)', fontsize=12)
    ax.set_xlabel('Method'); ax.set_ylabel('Average Turnover (%)')
    ax.grid(alpha=0.3, axis='y')
    plt.tight_layout()
    fig.savefig(_OUT / 'figure3_turnover.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved figure3_turnover.png")
    print(to_df.describe().round(1).to_string())


In [ ]:
import pandas as pd, matplotlib.pyplot as plt, matplotlib.dates as mdates
from pathlib import Path

_ROOT = Path.cwd()
while _ROOT != _ROOT.parent and not (_ROOT / 'src').exists():
    _ROOT = _ROOT.parent
_OUT = _ROOT / 'src/data/processed/phase1_full_replication'

_r = _OUT / 'figs345_retention.csv'
if not _r.exists():
    print("figs345_retention.csv not found. Run: python scripts/run_full_backtest.py")
else:
    ret_df = pd.read_csv(_r, index_col=0, parse_dates=True)
    ret_df.columns = ['Matching', 'Baseline']

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(ret_df.index, ret_df['Matching'], label='Matching', color='steelblue',  lw=1.5)
    ax.plot(ret_df.index, ret_df['Baseline'], label='Baseline', color='darkorange', lw=1.5)
    ax.set_title('Figure 4: Monthly Pair Retention vs Time', fontsize=12)
    ax.set_xlabel('Date'); ax.set_ylabel('% Retention (Jaccard)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.tick_params(axis='x', rotation=30)
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    fig.savefig(_OUT / 'figure4_retention.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved figure4_retention.png  (Matching mean={ret_df['Matching'].mean():.1f}%, Baseline mean={ret_df['Baseline'].mean():.1f}%)")


In [ ]:
import pandas as pd, matplotlib.pyplot as plt, matplotlib.dates as mdates
from pathlib import Path

_ROOT = Path.cwd()
while _ROOT != _ROOT.parent and not (_ROOT / 'src').exists():
    _ROOT = _ROOT.parent
_OUT = _ROOT / 'src/data/processed/phase1_full_replication'

_c = _OUT / 'figs345_concentration.csv'
if not _c.exists():
    print("figs345_concentration.csv not found. Run: python scripts/run_full_backtest.py")
else:
    conc_df = pd.read_csv(_c, index_col=0, parse_dates=True)

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(conc_df.index, conc_df['Baseline'], color='steelblue', lw=1.2)
    ax.fill_between(conc_df.index, conc_df['Baseline'], alpha=0.15, color='steelblue')
    ax.set_title('Figure 5: Single-Stock Concentration vs Time (Baseline Portfolio)', fontsize=12)
    ax.set_xlabel('Date'); ax.set_ylabel('Concentration')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
    ax.tick_params(axis='x', rotation=45); ax.grid(alpha=0.3)
    plt.tight_layout()
    fig.savefig(_OUT / 'figure5_concentration.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved figure5_concentration.png  (max={conc_df['Baseline'].max():.1f}, mean={conc_df['Baseline'].mean():.2f})")


In [ ]:
import pandas as pd, matplotlib.pyplot as plt, matplotlib.dates as mdates
from pathlib import Path

_ROOT = Path.cwd()
while _ROOT != _ROOT.parent and not (_ROOT / 'src').exists():
    _ROOT = _ROOT.parent
_OUT = _ROOT / 'src/data/processed/phase1_full_replication'

_diag_path = _OUT / 'diagnostics.csv'
if not _diag_path.exists():
    print(f"diagnostics.csv not found. Run: python scripts/run_full_backtest.py")
else:
    ok = pd.read_csv(_diag_path, parse_dates=['reb_date'])
    ok = ok[ok['status'] == 'ok'].copy()

    fig, axes = plt.subplots(2, 2, figsize=(15, 9))
    fig.suptitle('Backtest Diagnostics', fontsize=13)

    axes[0, 0].plot(ok['reb_date'], ok['n_eligible'], color='steelblue', lw=1.5)
    axes[0, 0].set_title('Eligible Tickers'); axes[0, 0].set_ylabel('Count')

    axes[0, 1].plot(ok['reb_date'], ok['n_edges'], color='steelblue', lw=1.5)
    axes[0, 1].set_title('Cointegrated Pairs (edges)'); axes[0, 1].set_ylabel('Count')

    axes[1, 0].plot(ok['reb_date'], ok['n_pairs_M'], label='Matching', color='steelblue', lw=1.5)
    axes[1, 0].plot(ok['reb_date'], ok['n_pairs_B'], label='Baseline', color='darkorange', lw=1.5, ls='--')
    axes[1, 0].set_title('Selected Pairs'); axes[1, 0].set_ylabel('Count'); axes[1, 0].legend()

    axes[1, 1].bar(ok['reb_date'], ok['iter_sec'], width=20, color='steelblue', alpha=0.7)
    axes[1, 1].set_title('Computation Time per Rebalance (sec)'); axes[1, 1].set_ylabel('Seconds')

    for ax in axes.flat:
        ax.xaxis.set_major_locator(mdates.YearLocator())
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
        ax.tick_params(axis='x', rotation=30); ax.grid(alpha=0.3)

    plt.tight_layout()
    fig.savefig(_OUT / 'figure_diagnostics.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved figure_diagnostics.png  ({len(ok)} rebalances shown)")


## 7. Save Outputs

In [ ]:
# Guard: only save if the backtest produced real results (not stale empty data)
_ok = ('ret_gross_M' in dir() and
       isinstance(ret_gross_M, pd.Series) and
       ret_gross_M.dropna().shape[0] > 100)

if not _ok:
    print("Skipping save — backtest_loop hasn't run this session. Run it first.")
else:
    returns_df = pd.DataFrame({
        'gross_matching': ret_gross_M, 'net_matching': ret_net_M,
        'gross_baseline': ret_gross_B, 'net_baseline': ret_net_B,
        'benchmark': benchmark_ret,    'rf_daily': risk_free_daily,
    })
    returns_df.to_parquet(OUT / 'strategy_returns.parquet')
    diag_df.to_csv(OUT / 'diagnostics.csv', index=False)
    if not edges_df.empty:  edges_df.to_parquet(OUT / 'candidate_edges.parquet', index=False)
    if not sel_M_df.empty:  sel_M_df.to_parquet(OUT / 'selected_pairs_matching.parquet', index=False)
    if not sel_B_df.empty:  sel_B_df.to_parquet(OUT / 'selected_pairs_baseline.parquet', index=False)
    perf_df.to_csv(OUT / 'table_performance.csv', index=False)
    corr_df.to_csv(OUT / 'table_correlation.csv')
    print('Saved to', OUT)
    print(perf_df.to_string(index=False))
